# Practice Lab: Embeddings & Vector Stores (Lesson 8.3)

Module 8 · 8 exercises · MTEB + Matryoshka + HNSW + hybrid search + Indic

Exercises 1, 2, 4, 5, 7 run locally (numpy only).


# Lesson 8.3: Embeddings & Vector Stores

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 4, 5, 7 run **locally** (numpy). Ex 3, 6, 8 are architecture/design.


In [ ]:
import numpy as np
import hashlib, math, time, json
from collections import Counter


---
## Exercise 1 (Easy): Embedding Model Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
models = [
    ("Gemini Emb 001", 68.32, "768-3072", 0.15, "Best retrieval"),
    ("Qwen3-Emb-8B", 70.58, "32-4096", 0.001, "#1 OSS Apache2"),
    ("Voyage 3.5", 66.80, "256-2048", 0.06, "Best quality/price"),
    ("Cohere v4", 65.20, "256-1536", 0.12, "Multimodal"),
    ("OpenAI 3-small", 62.26, "256-1536", 0.02, "Cheapest API"),
    ("BGE-M3", 63.00, "1024", 0.001, "Dense+sparse MIT"),
    ("jina-v3", 65.52, "32-1024", 0.001, "5 LoRA adapters"),
]

print(f"{'Model':<20} {'MTEB':<8} {'Dims':<12} {'$/M':<8}")
print("=" * 50)
for name, mteb, dims, cost, feat in sorted(models, key=lambda x: -x[1]):
    print(f"  {name:<18} {mteb:<8.2f} {dims:<12} ${cost:<6.3f}")

print(f"\nMonthly Cost (50M tokens):")
for name, _, _, cost, _ in models:
    print(f"  {name:<18} ${cost*50:.2f}")

print(f"\nSelf-host breakeven: ~100M tok/mo on A100 ($1,800/mo)")
print(f"Recommendations: budget=OpenAI 3-small, quality=Gemini, OSS=Qwen3")
print(f"Critical: switching models = re-embed entire corpus")


</details>


---
## Exercise 2 (Easy): Matryoshka Dimension Sweep


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib

def matryoshka_embed(text, full_dim=1024):
    h = hashlib.sha256(text.lower().encode()).digest() * 40
    vec = np.array([b/255.0 for b in h[:full_dim]])
    kw = {"refund":0,"price":1,"cost":2,"course":3,"live":4,"emi":6,"cert":7,"python":8}
    for w, i in kw.items():
        if w in text.lower() and i < full_dim: vec[i] += 1.0
    return vec / np.linalg.norm(vec)

docs = [
    "Refund policy full refund within 7 days 50 percent from 8-30 days",
    "GenAI course costs 14999 rupees lifetime access certificate",
    "Live classes daily 7 PM IST YouTube recordings 2 hours",
    "EMI available Razorpay 2500 rupees per month 6 months",
]
queries = [("refund policy","refund"),("course price","14999"),("live class","7 PM"),("EMI options","emi")]

full_embs = [matryoshka_embed(d) for d in docs]
full_qembs = [matryoshka_embed(q) for q, _ in queries]

print(f"Matryoshka Dimension Sweep:")
print(f"{'Dims':<8} {'Recall@3':<12} {'Storage/vec':<14} {'Savings'}")
print("=" * 50)

for d in [32, 64, 128, 256, 512, 768, 1024]:
    te = [e[:d] for e in full_embs]
    tq = [e[:d] for e in full_qembs]
    hits = 0
    for qi, (q, exp) in enumerate(queries):
        scores = sorted([(i, np.dot(tq[qi], e)) for i, e in enumerate(te)], key=lambda x: -x[1])[:3]
        if exp.lower() in " ".join(docs[i] for i, _ in scores).lower(): hits += 1
    recall = hits / len(queries)
    storage = d * 4
    savings = (1 - storage / (1024*4)) * 100
    print(f"  {d:<6} {recall:<12.0%} {storage:<14} {savings:.0f}%")

print(f"\nStorage at 100M vectors:")
for d in [256, 768, 3072]:
    print(f"  {d}d: {100_000_000 * d * 4 / 1e9:.0f} GB")
print(f"Use dot product on truncated (not cosine)")


</details>


---
## Exercise 4 (Medium): HNSW Parameter Tuning


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np

def sim_recall(ef, M=16):
    base = 0.70 + 0.05 * np.log2(M / 8)
    recall = min(0.999, base + 0.10 * np.log2(ef / 10))
    latency = 0.5 + ef * 0.015
    return round(recall, 4), round(latency, 2)

print("HNSW efSearch Tuning (M=16, 1M vectors):")
print(f"{'efSearch':<10} {'Recall':<10} {'Latency':<12} {'Notes'}")
print("=" * 45)
for ef in [10, 25, 50, 100, 200, 500, 1000]:
    r, l = sim_recall(ef)
    notes = "Too low" if r < 0.85 else "Good" if r < 0.95 else "High recall" if r < 0.99 else "Near-perfect"
    print(f"  {ef:<8} {r:<10.1%} {l:<12.1f}ms {notes}")

print(f"\nM Parameter Memory (1M vectors, 768d):")
N, D = 1_000_000, 768
for M in [8, 16, 32, 64]:
    graph = N * (M * 2 * 4 + 8)
    vecs = N * D * 4
    total = (graph + vecs) / 1e9
    print(f"  M={M:<4} {total:.1f}GB")

print(f"\nFormula: Graph=N*(M*2*4+8), Vectors=N*D*4")
print(f"Strategy: efSearch first (no rebuild) -> M only if needed")


</details>


---
## Exercise 5 (Medium): Hybrid Search Implementation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib, math
from collections import Counter

docs = ["RAG retrieval augmented generation combines search with LLMs",
        "BM25 probabilistic ranking function term frequency document length",
        "HNSW navigable small world graph approximate nearest neighbor",
        "Cosine similarity angle between vectors embedding space",
        "Qdrant Rust vector database filterable HNSW binary quantization",
        "Refund policy full refund 7 days 50 percent 8-30 days",
        "GenAI course 14999 rupees lifetime access certificate GCP",
        "Live classes 7 PM IST YouTube recordings 2 hours daily"]

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in corpus]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for doc in self.docs:
            for w in set(doc): self.df[w] += 1
        self.n = len(self.docs)
    def rank(self, q, k=5):
        qt = q.lower().split(); scores = []
        for i, doc in enumerate(self.docs):
            s, tf = 0, Counter(doc)
            for t in qt:
                if t not in self.df: continue
                idf = math.log((self.n-self.df[t]+0.5)/(self.df[t]+0.5)+1)
                s += idf * tf.get(t,0)*(self.k1+1) / (tf.get(t,0)+self.k1*(1-self.b+self.b*len(doc)/self.avg_dl))
            scores.append((i, s))
        return sorted(scores, key=lambda x: -x[1])[:k]

def fake_embed(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"rag":0,"bm25":1,"hnsw":2,"qdrant":4,"refund":6,"price":7,"live":9}
    for w, i in kw.items():
        if w in t.lower().split() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def dense_rank(q, corpus, k=5):
    qe = fake_embed(q)
    return sorted([(i, np.dot(qe, fake_embed(d))/(np.linalg.norm(qe)*np.linalg.norm(fake_embed(d)))) for i, d in enumerate(corpus)], key=lambda x: -x[1])[:k]

def rrf(rankings, k=60):
    scores = {}
    for r in rankings:
        for rank, (idx, _) in enumerate(r):
            scores[idx] = scores.get(idx, 0) + 1.0/(k+rank+1)
    return sorted(scores.items(), key=lambda x: -x[1])

def alpha_merge(dense_scores, bm25_scores, alpha=0.5):
    """alpha=1.0 = all dense, alpha=0.0 = all BM25"""
    combined = {}
    d_max = max(s for _, s in dense_scores) or 1
    b_max = max(s for _, s in bm25_scores) or 1
    for idx, s in dense_scores:
        combined[idx] = combined.get(idx, 0) + alpha * (s / d_max)
    for idx, s in bm25_scores:
        combined[idx] = combined.get(idx, 0) + (1 - alpha) * (s / b_max)
    return sorted(combined.items(), key=lambda x: -x[1])

bm25 = BM25(docs)
queries = [("BM25 ranking", "keyword"), ("money back refund", "semantic"),
           ("Qdrant HNSW", "mixed"), ("course price EMI", "mixed")]

print("Hybrid Search:")
print("=" * 55)
for q, qt in queries:
    br = bm25.rank(q); dr = dense_rank(q, docs); hr = rrf([br, dr])[:3]
    print(f"  Q: {q} ({qt})")
    print(f"    BM25:   [{br[0][0]}] {docs[br[0][0]][:45]}...")
    print(f"    Dense:  [{dr[0][0]}] {docs[dr[0][0]][:45]}...")
    print(f"    Hybrid: [{hr[0][0]}] {docs[hr[0][0]][:45]}...")

print(f"\nAlpha Tuning (query: 'refund policy'):")
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    merged = alpha_merge(dense_rank("refund policy", docs), bm25.rank("refund policy"), alpha=alpha)
    top = merged[0]
    label = ("BM25 only" if alpha == 0 else "Tech docs" if alpha == 0.3 else
             "Balanced" if alpha == 0.5 else "Natural lang" if alpha == 0.7 else "Dense only")
    print(f"  alpha={alpha:.1f} ({label:<12}): [{top[0]}] {docs[top[0]][:45]}...")

print(f"\nRRF: score=sum(1/(60+rank)). Impact: +26% NDCG")


</details>


---
## Exercise 7 (Challenge): Two-Stage Matryoshka Retrieval


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib, time

def mat_embed(text, dim=1024):
    h = hashlib.sha256(text.lower().encode()).digest() * 40
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"refund":0,"price":1,"cost":2,"course":3,"live":4,"emi":6}
    for w, i in kw.items():
        if w in text.lower() and i < dim: vec[i] += 1.0
    return vec / np.linalg.norm(vec)

base_docs = ["Refund policy full refund 7 days 50 percent 8-30 days",
             "GenAI course 14999 rupees lifetime access cert",
             "Live classes 7 PM IST YouTube recordings",
             "EMI Razorpay 2500 per month 6 months",
             "14 modules 146 hours Python GCP deployment",
             "Prerequisites Python math no ML needed",
             "Discord community weekly live sessions",
             "85 completion 92 placement 4.8 stars"]
docs = base_docs * 125
embs = [mat_embed(d) for d in docs]
q_emb = mat_embed("refund policy money back")

# Single-stage
t0 = time.time()
full_scores = sorted([(i, np.dot(q_emb, e)) for i, e in enumerate(embs)], key=lambda x: -x[1])
t1 = time.time()
single_top = [i for i, _ in full_scores[:10]]

# Two-stage
t2 = time.time()
sq = q_emb[:256]; se = [e[:256] for e in embs]
stage1 = sorted([(i, np.dot(sq, e)) for i, e in enumerate(se)], key=lambda x: -x[1])[:200]
stage2 = sorted([(i, np.dot(q_emb, embs[i])) for i, _ in stage1], key=lambda x: -x[1])
t3 = time.time()
two_top = [i for i, _ in stage2[:10]]

overlap = len(set(single_top) & set(two_top))
print(f"Two-Stage Matryoshka ({len(docs)} docs):")
print(f"  Single (1024d): {(t1-t0)*1000:.1f}ms -> {docs[single_top[0]][:40]}...")
print(f"  Two-stage:      {(t3-t2)*1000:.1f}ms -> {docs[two_top[0]][:40]}...")
print(f"  Overlap: {overlap}/10 ({overlap*10}%)")

print(f"\nStorage at 100M vectors:")
for d in [256, 768, 3072]: print(f"  {d}d: {100_000_000*d*4/1e9:.0f}GB")
print(f"  256d saves {(1-256/3072)*100:.0f}% vs 3072d")
print(f"\nVersioning: blue-green with is_current flag")


</details>


---
## Exercise 3 (Medium): Vector Database Migration
Architecture/design. See practice-lab-8_3.html.


In [ ]:
# Exercise 3: Vector Database Migration
pass


---
## Exercise 6 (Challenge): Metadata Pre-Filtering Benchmark
Architecture/design. See practice-lab-8_3.html.


In [ ]:
# Exercise 6: Metadata Pre-Filtering Benchmark
pass


---
## Exercise 8 (Challenge): Multilingual Indic RAG
Architecture/design. See practice-lab-8_3.html.


In [ ]:
# Exercise 8: Multilingual Indic RAG
pass
